In the previous post, I [read a payment transaction](https://arjunrajkumar.github.io/deconstructing-a-payment-transaction/) byte by byte. Here I want to verify the signature on its first input.

This is input 1 of 11 from the earlier post. It spends output index `0` of a previous transaction.. The input carries an **unlocking script** (`scriptSig`) with the spender’s signature and public key.

```text
Input 1 of 11 (byte offsets 5 through 184, counting from 0)
  Previous TXID : 3fd4cb75e272a5e68524d15df2d16a7b926d4998c2f1bc60a6f76083eeaba404
  Output index  : 0  (bytes: 00 00 00 00)
  Script length : 139 bytes  (encoded as: 8b)
  Unlocking script bytes:
    48 30 45 02 21 00 c6 d5 92 90 df 93 4b ac 08 bc
    41 24 b0 61 54 ee 21 39 4d 37 ec 95 8d 14 51 29
    a8 85 db d6 3e ab 02 20 03 62 51 12 0e 4c ea cc
    8e 9f f1 35 85 04 99 45 c7 db db 85 92 1a 6e 07
    7a 80 c7 e4 10 93 1f 4a 01 41 04 f3 ce 89 48 c5
    92 69 0c b3 6a 7e 6e 31 53 49 27 56 9f 41 e4 5c
    77 fb db 95 fd b0 e2 35 71 7a 29 5b 2a 28 1c 04
    61 91 39 c4 53 dd 91 39 e9 51 08 63 25 b4 02 1b
    d2 8d a6 fc 3d a8 69 43 cd 04 cf
  Sequence      : 4294967295  (bytes: ff ff ff ff)
```

Apart from this input, I also need the output that this input spends. Its transaction ID (`previous_txid`) tells me which transaction to look up. Its output index (`previous_output_index`) tells me which output to read, and the locking script `scriptPubKey` will tell me the spending conditions that the spender must match to spend the funds.

So, to summarize the current inputs unlocking script should match the previous transactions lockcing script.

My pruned node discards old block data to save space. I used a block explorer to find the block that contains the previous transaction. That is block has `00000000000007fe3e6060ec0c6ddb6790c77a8dc06eea4575a2408eebbf07d0`. As the full block is missing from my node, I can request it from a connected peer with `getblockfrompeer`. Once the block is available, I can use `getrawtransaction` to look up the previous transaction using its ID and the block hash. I saved that reply in `previous_tx` below.

In [21]:
previous_txid = "3fd4cb75e272a5e68524d15df2d16a7b926d4998c2f1bc60a6f76083eeaba404"
previous_tx_block_hash = "00000000000007fe3e6060ec0c6ddb6790c77a8dc06eea4575a2408eebbf07d0"
previous_output_index = 0

unlocking_script_bytes = bytes.fromhex("""
    48 30 45 02 21 00 c6 d5 92 90 df 93 4b ac 08 bc
    41 24 b0 61 54 ee 21 39 4d 37 ec 95 8d 14 51 29
    a8 85 db d6 3e ab 02 20 03 62 51 12 0e 4c ea cc
    8e 9f f1 35 85 04 99 45 c7 db db 85 92 1a 6e 07
    7a 80 c7 e4 10 93 1f 4a 01 41 04 f3 ce 89 48 c5
    92 69 0c b3 6a 7e 6e 31 53 49 27 56 9f 41 e4 5c
    77 fb db 95 fd b0 e2 35 71 7a 29 5b 2a 28 1c 04
    61 91 39 c4 53 dd 91 39 e9 51 08 63 25 b4 02 1b
    d2 8d a6 fc 3d a8 69 43 cd 04 cf
""")

print(f"Unlocking script: {len(unlocking_script_bytes)} bytes")

Unlocking script: 139 bytes


In [23]:
# I got previous_tx via bitcoin cli
# !bitcoin-cli getblockfrompeer {previous_tx_block_hash} 0
# !bitcoin-cli getrawtransaction {previous_txid} true {previous_tx_block_hash}

previous_tx = {
  "in_active_chain": True,
  "txid": "3fd4cb75e272a5e68524d15df2d16a7b926d4998c2f1bc60a6f76083eeaba404",
  "hash": "3fd4cb75e272a5e68524d15df2d16a7b926d4998c2f1bc60a6f76083eeaba404",
  "version": 1,
  "size": 225,
  "vsize": 225,
  "weight": 900,
  "locktime": 0,
  "vin": [
    {
      "txid": "4e598fea2110bc7f7b1c433db4f9a489ff1a2ba68874000fe9ac8da186c318b7",
      "vout": 1,
      "scriptSig": {
        "asm": "3046022100dbff78061770bf76984c4ea7de28124b3443b96247448d9138710bbb81572cef022100af268d05fbffa315c51811c4c6baf747b6083d62eaaf12762530997c7250700b[ALL] 04337d915c620beac43c8d358158f777ea4d040099e3deb67b63a8e61361ff4f43847b05b5ee971fa3aeb748d3e553dce24b0978831984d9a0b1dbdb648438c34c",
        "hex": "493046022100dbff78061770bf76984c4ea7de28124b3443b96247448d9138710bbb81572cef022100af268d05fbffa315c51811c4c6baf747b6083d62eaaf12762530997c7250700b014104337d915c620beac43c8d358158f777ea4d040099e3deb67b63a8e61361ff4f43847b05b5ee971fa3aeb748d3e553dce24b0978831984d9a0b1dbdb648438c34c"
      },
      "sequence": 4294967295
    }
  ],
  "vout": [
    {
      "value": 50000.00000000,
      "n": 0,
      "scriptPubKey": {
        "asm": "OP_DUP OP_HASH160 fef5cca40eaf93988c09ee70728e139f65f009ff OP_EQUALVERIFY OP_CHECKSIG",
        "desc": "addr(1QF76AaXFWmMqeiKsRQvQNnz4AhJ91tkwB)#7tjw2xwp",
        "hex": "76a914fef5cca40eaf93988c09ee70728e139f65f009ff88ac",
        "address": "1QF76AaXFWmMqeiKsRQvQNnz4AhJ91tkwB",
        "type": "pubkeyhash"
      }
    }
  ],
  "hex": "0100000001b718c386a18dace90f007488a62b1aff89a4f9b43d431c7b7fbc1021ea8f594e010000008c493046022100dbff78061770bf76984c4ea7de28124b3443b96247448d9138710bbb81572cef022100af268d05fbffa315c51811c4c6baf747b6083d62eaaf12762530997c7250700b014104337d915c620beac43c8d358158f777ea4d040099e3deb67b63a8e61361ff4f43847b05b5ee971fa3aeb748d3e553dce24b0978831984d9a0b1dbdb648438c34cffffffff01005039278c0400001976a914fef5cca40eaf93988c09ee70728e139f65f009ff88ac00000000",
  "blockhash": "00000000000007fe3e6060ec0c6ddb6790c77a8dc06eea4575a2408eebbf07d0",
  "confirmations": 720551,
  "time": 1315755730,
  "blocktime": 1315755730
}

previous_output = previous_tx["vout"][previous_output_index]
previous_output_value_btc = previous_output["value"]
locking_script_hex = previous_output["scriptPubKey"]["hex"]
previous_output

{'value': 50000.0,
 'n': 0,
 'scriptPubKey': {'asm': 'OP_DUP OP_HASH160 fef5cca40eaf93988c09ee70728e139f65f009ff OP_EQUALVERIFY OP_CHECKSIG',
  'desc': 'addr(1QF76AaXFWmMqeiKsRQvQNnz4AhJ91tkwB)#7tjw2xwp',
  'hex': '76a914fef5cca40eaf93988c09ee70728e139f65f009ff88ac',
  'address': '1QF76AaXFWmMqeiKsRQvQNnz4AhJ91tkwB',
  'type': 'pubkeyhash'}}

So, now that the setup is complete we have all the data to verify that the unlocking script satisfies the locking script.